In [1]:
# ============================================================
# CELL 1 ▸ Imports & Config (same as original)
# ============================================================
import subprocess, sys

for pkg in ['librosa','soundfile','audioread','numpy','pandas',
            'scipy','tqdm']:
    try:   __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os, gc, glob, re, time, warnings, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from scipy import stats
import librosa
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
import multiprocessing
CPU_COUNT = multiprocessing.cpu_count()

print(f'librosa  {librosa.__version__}')
print(f'numpy    {np.__version__}')
print(f'CPUs     {CPU_COUNT}')
print('Imports complete.')

# --------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------
# Set paths – adjust if you put the dataset elsewhere
# The existing per‑class CSVs are stored in the dataset "CompletedExtraction"
# We'll read them from the input directory.
INPUT_DATA_DIR = '/kaggle/input/datasets/mlfarhan/completedextraction'   # name of your uploaded dataset
# If the files are in the working directory (e.g., after previous run), you can set:
# INPUT_DATA_DIR = '/kaggle/working/part2_outputs/per_class_csv'

WORK_DIR  = '/kaggle/working'
OUT_DIR   = os.path.join(WORK_DIR, 'part2_outputs')
PART_DIR  = os.path.join(OUT_DIR,  'per_class_csv')
for d in [OUT_DIR, PART_DIR]:
    os.makedirs(d, exist_ok=True)

# Audio params (must match original)
SR           = 22050
SEG_DUR      = 10
SEG_SAMPLES  = SR * SEG_DUR
HOP_LENGTH   = 512
N_MELS       = 128
N_MFCC       = 40
N_FFT        = 2048
SEED         = 42

# Speed settings
N_WORKERS    = max(1, CPU_COUNT)
BATCH_FLUSH  = 50
MIN_FEATURES = 200

AUDIO_EXT = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac')

CLASS_NAMES = [
    'Bangla_Band_Collection', 'Bangla_ancholik',
    'Bangla_baby_rhymes',     'Bangla_rabindra_sangeet',
    'Bangla_romantic',        'Classic_Songs_',
    'Islamic_Melody',         'Nazrul_Geeti_',
]
COARSE_MAP = {
    'Bangla_Band_Collection'  : 'Contemporary_Pop',
    'Bangla_ancholik'         : 'Folk_Regional',
    'Bangla_baby_rhymes'      : 'Folk_Regional',
    'Bangla_rabindra_sangeet' : 'Classical_Tagore',
    'Bangla_romantic'         : 'Contemporary_Pop',
    'Classic_Songs_'          : 'Classical_Tagore',
    'Islamic_Melody'          : 'Devotional_Spiritual',
    'Nazrul_Geeti_'           : 'Devotional_Spiritual',
}

print('Configuration loaded.')
print(f'  SR={SR}  SEG_DUR={SEG_DUR}s  N_MFCC={N_MFCC}  N_WORKERS={N_WORKERS}')
print(f'  Input dir (existing CSVs): {INPUT_DATA_DIR}')
print(f'  Output dir (new CSVs)    : {PART_DIR}')

librosa  0.11.0
numpy    2.4.6
CPUs     4
Imports complete.
Configuration loaded.
  SR=22050  SEG_DUR=10s  N_MFCC=40  N_WORKERS=4
  Input dir (existing CSVs): /kaggle/input/datasets/mlfarhan/completedextraction
  Output dir (new CSVs)    : /kaggle/working/part2_outputs/per_class_csv


In [2]:
# ============================================================
# CELL 2 ▸ Dataset Discovery (only for missing classes)
# ============================================================
# First, locate the raw audio dataset root (same as original)
AUDIO_ROOT = '/kaggle/input/datasets/mlfarhan/audio-classification/audio genre classification'

def find_root(base, cls_list):
    for root, dirs, _ in os.walk(base):
        hits = sum(1 for d in dirs
                   if any(c.strip('_ ').lower() in d.strip('_ ').lower()
                          for c in cls_list))
        if hits >= 4: return root
    return base

DATASET_ROOT = find_root(AUDIO_ROOT, CLASS_NAMES)
print(f'Dataset root: {DATASET_ROOT}')

# Build file list for all classes (needed for remaining ones)
class_files = {}
for cls in CLASS_NAMES:
    folder = os.path.join(DATASET_ROOT, cls)
    if not os.path.isdir(folder):
        for d in os.listdir(DATASET_ROOT):
            if cls.strip('_ ').lower() in d.strip('_ ').lower():
                folder = os.path.join(DATASET_ROOT, d); break
    files = []
    if os.path.isdir(folder):
        for ext in AUDIO_EXT:
            files += glob.glob(os.path.join(folder, f'*{ext}'))
            files += glob.glob(os.path.join(folder, f'*{ext.upper()}'))
    class_files[cls] = sorted(set(files))

print('\nDiscovered audio files:')
for cls in CLASS_NAMES:
    print(f'  {cls:<35} {len(class_files[cls]):>4} files')
total = sum(len(v) for v in class_files.values())
print(f'  {"─"*35}  {"─"*4}')
print(f'  {"TOTAL":<35} {total:>4}')

# --------------------------------------------------------------
# Determine which classes already have CSVs (from the uploaded dataset)
# --------------------------------------------------------------
completed_classes = []
for cls in CLASS_NAMES:
    csv_path = os.path.join(INPUT_DATA_DIR, f'feat_{cls}.csv')
    if os.path.exists(csv_path):
        completed_classes.append(cls)
        # Also copy the file to our working PART_DIR for later merging
        import shutil
        shutil.copy(csv_path, os.path.join(PART_DIR, f'feat_{cls}.csv'))
        print(f'  ✅ Found existing CSV for {cls} → copied to working dir')

missing_classes = [cls for cls in CLASS_NAMES if cls not in completed_classes]
print(f'\nClasses to process: {missing_classes}')

Dataset root: /kaggle/input/datasets/mlfarhan/audio-classification/audio genre classification

Discovered audio files:
  Bangla_Band_Collection               287 files
  Bangla_ancholik                      169 files
  Bangla_baby_rhymes                   153 files
  Bangla_rabindra_sangeet              152 files
  Bangla_romantic                      291 files
  Classic_Songs_                       245 files
  Islamic_Melody                       319 files
  Nazrul_Geeti_                        254 files
  ───────────────────────────────────  ────
  TOTAL                               1870
  ✅ Found existing CSV for Bangla_Band_Collection → copied to working dir
  ✅ Found existing CSV for Bangla_ancholik → copied to working dir
  ✅ Found existing CSV for Bangla_baby_rhymes → copied to working dir
  ✅ Found existing CSV for Bangla_rabindra_sangeet → copied to working dir
  ✅ Found existing CSV for Bangla_romantic → copied to working dir
  ✅ Found existing CSV for Classic_Songs_ → copie

In [3]:
# ============================================================
# CELL 3 ▸ Feature Extraction Functions (FIXED)
# ============================================================
def load_audio(path, sr=SR):
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        return y
    except Exception:
        return None

def segment_audio(y, sr=SR, duration=SEG_DUR):
    seg_len = sr * duration
    return [y[s:s+seg_len]
            for s in range(0, len(y)-seg_len+1, seg_len)
            if np.max(np.abs(y[s:s+seg_len])) > 1e-4]

def extract_features(y, sr=SR):
    f = {}

    # ---------- MFCC ----------
    try:
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                     hop_length=HOP_LENGTH, n_fft=N_FFT)
        d1 = librosa.feature.delta(mfcc)
        d2 = librosa.feature.delta(mfcc, order=2)
        for i in range(N_MFCC):
            f[f'mfcc_m{i}']   = float(np.mean(mfcc[i]))
            f[f'mfcc_s{i}']   = float(np.std(mfcc[i]))
            f[f'dmfcc_m{i}']  = float(np.mean(d1[i]))
            f[f'ddmfcc_m{i}'] = float(np.mean(d2[i]))
    except Exception:
        for i in range(N_MFCC):
            f[f'mfcc_m{i}'] = 0.0
            f[f'mfcc_s{i}'] = 0.0
            f[f'dmfcc_m{i}'] = 0.0
            f[f'ddmfcc_m{i}'] = 0.0

    # ---------- Chroma ----------
    try:
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=HOP_LENGTH)
        for i in range(12):
            f[f'chroma_m{i}'] = float(np.mean(chroma[i]))
            f[f'chroma_s{i}'] = float(np.std(chroma[i]))
    except Exception:
        for i in range(12):
            f[f'chroma_m{i}'] = 0.0
            f[f'chroma_s{i}'] = 0.0

    # ---------- Spectral descriptors ----------
    try:
        sc  = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH)[0]
        sb  = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=HOP_LENGTH)[0]
        sro = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP_LENGTH,
                                                roll_percent=0.85)[0]
        sfl = librosa.feature.spectral_flatness(y=y, hop_length=HOP_LENGTH)[0]
        f.update({'sc_mean': float(np.mean(sc)),   'sc_std': float(np.std(sc)),
                  'sb_mean': float(np.mean(sb)),   'sb_std': float(np.std(sb)),
                  'sro_mean': float(np.mean(sro)), 'sro_std': float(np.std(sro)),
                  'sfl_mean': float(np.mean(sfl)), 'sfl_std': float(np.std(sfl))})
    except Exception:
        for key in ['sc_mean','sc_std','sb_mean','sb_std',
                    'sro_mean','sro_std','sfl_mean','sfl_std']:
            f[key] = 0.0

    # ---------- Spectral contrast (7 bands with padding) ----------
    try:
        sct = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=HOP_LENGTH,
                                                n_bands=7)
        n_bands = sct.shape[0]
    except Exception:
        try:
            sct = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=HOP_LENGTH)
            n_bands = sct.shape[0]
        except Exception:
            sct = None
            n_bands = 0

    for i in range(7):
        if sct is not None and i < n_bands:
            f[f'contrast_m{i}'] = float(np.mean(sct[i]))
            f[f'contrast_s{i}'] = float(np.std(sct[i]))
        else:
            f[f'contrast_m{i}'] = 0.0
            f[f'contrast_s{i}'] = 0.0

    # ---------- Spectral entropy ----------
    try:
        stft_mag = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
        ps       = stft_mag**2
        ps_norm  = ps / (ps.sum(axis=0, keepdims=True) + 1e-10)
        entropy  = -np.sum(ps_norm * np.log2(ps_norm + 1e-10), axis=0)
        f['spec_entropy_mean'] = float(np.mean(entropy))
        f['spec_entropy_std']  = float(np.std(entropy))
    except Exception:
        f['spec_entropy_mean'] = 0.0
        f['spec_entropy_std']  = 0.0

    # ---------- Zero-crossing rate ----------
    try:
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)[0]
        f.update({'zcr_mean': float(np.mean(zcr)),
                  'zcr_std' : float(np.std(zcr)),
                  'zcr_skew': float(stats.skew(zcr)),
                  'zcr_kurt': float(stats.kurtosis(zcr))})
    except Exception:
        for key in ['zcr_mean','zcr_std','zcr_skew','zcr_kurt']:
            f[key] = 0.0

    # ---------- RMS ----------
    try:
        rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)[0]
        f.update({'rms_mean'     : float(np.mean(rms)),
                  'rms_std'      : float(np.std(rms)),
                  'rms_skew'     : float(stats.skew(rms)),
                  'rms_kurt'     : float(stats.kurtosis(rms)),
                  'rms_dyn_range': float(np.max(rms) - np.min(rms))})
    except Exception:
        for key in ['rms_mean','rms_std','rms_skew','rms_kurt','rms_dyn_range']:
            f[key] = 0.0

    # ---------- Tonnetz ----------
    try:
        yh = librosa.effects.harmonic(y)
        tn = librosa.feature.tonnetz(y=yh, sr=sr)
        n_ton = tn.shape[0]
        for i in range(6):
            if i < n_ton:
                f[f'tonnetz_m{i}'] = float(np.mean(tn[i]))
                f[f'tonnetz_s{i}'] = float(np.std(tn[i]))
            else:
                f[f'tonnetz_m{i}'] = 0.0
                f[f'tonnetz_s{i}'] = 0.0
    except Exception:
        for i in range(6):
            f[f'tonnetz_m{i}'] = 0.0
            f[f'tonnetz_s{i}'] = 0.0

    # ---------- Mel spectrogram stats ----------
    try:
        mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                                 hop_length=HOP_LENGTH, n_fft=N_FFT)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        f.update({'mel_mean': float(np.mean(mel_db)),
                  'mel_std' : float(np.std(mel_db)),
                  'mel_skew': float(stats.skew(mel_db.flatten())),
                  'mel_kurt': float(stats.kurtosis(mel_db.flatten())),
                  'mel_min' : float(np.min(mel_db)),
                  'mel_max' : float(np.max(mel_db))})
    except Exception:
        for key in ['mel_mean','mel_std','mel_skew','mel_kurt','mel_min','mel_max']:
            f[key] = 0.0

    # ---------- Tempo ----------
    try:
        tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
        f['tempo'] = float(tempo)
        if len(beats) > 1:
            bt = librosa.frames_to_time(beats, sr=sr, hop_length=HOP_LENGTH)
            f['beat_strength'] = float(np.mean(np.diff(bt)))
        else:
            f['beat_strength'] = 0.0
    except Exception:
        f['tempo'] = 0.0
        f['beat_strength'] = 0.0

    return f

# Smoke test
_y = np.random.randn(SEG_SAMPLES).astype(np.float32)
_fv = extract_features(_y)
print(f'Smoke test — features extracted: {len(_fv)}')
assert len(_fv) == 237, f'Expected 237, got {len(_fv)}'
del _y, _fv
print('Feature extractor ready. (237 features confirmed)')

Smoke test — features extracted: 237
Feature extractor ready. (237 features confirmed)


In [4]:
# ============================================================
# CELL 4 ▸ Run Extraction for Missing Classes Only
# ============================================================
def process_file(fpath, cls, coarse):
    rows = []
    try:
        y = load_audio(fpath)
        if y is None:
            return rows, {'filepath':fpath,'cls':cls,'status':'load_error','n_segs':0}
        segs    = segment_audio(y)
        song_id = os.path.splitext(os.path.basename(fpath))[0]
        for s_idx, seg in enumerate(segs):
            fv = extract_features(seg)
            if len(fv) < MIN_FEATURES: continue
            row = {'filepath':fpath,'label':cls,'coarse':coarse,
                   'song_id':song_id,'seg_idx':s_idx,'seg_duration_s':SEG_DUR}
            row.update(fv)
            rows.append(row)
        del y; gc.collect()
        return rows, {'filepath':fpath,'cls':cls,'status':'ok','n_segs':len(rows)}
    except Exception as e:
        return rows, {'filepath':fpath,'cls':cls,'status':f'error:{e}','n_segs':0}

def extract_class(cls, coarse, files, part_dir, n_workers, batch):
    class_csv = os.path.join(part_dir, f'feat_{cls}.csv')
    header_written = False
    written = 0
    t0 = time.time()

    # Resume: if CSV already exists (from previous partial run), skip files already done
    done_fps = set()
    if os.path.exists(class_csv):
        try:
            done_fps = set(pd.read_csv(class_csv, usecols=['filepath'])['filepath'])
            written = sum(1 for _ in open(class_csv)) - 1
            header_written = True
            print(f'    [resume] {cls}: {len(done_fps)} files done, {written:,} rows saved')
        except Exception:
            pass

    remaining = [f for f in files if f not in done_fps]
    if not remaining:
        print(f'    [skip]   {cls}: already complete')
        return {'class':cls,'n_files':len(files),'n_segs':written,
                'errors':0,'elapsed_min':0}

    buffer = []; errors = 0
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(process_file, fp, cls, coarse): fp for fp in remaining}
        pbar = tqdm(as_completed(futures), total=len(remaining),
                    desc=f'  {cls[:30]:<30}', leave=False, unit='file')
        for fut in pbar:
            rows, log = fut.result()
            if 'error' in log['status']: errors += 1
            buffer.extend(rows)
            written += len(rows)
            if len(buffer) >= batch:
                chunk = pd.DataFrame(buffer)
                chunk.to_csv(class_csv, mode='a',
                             header=not header_written, index=False)
                header_written = True
                buffer = []
            pbar.set_postfix(segs=f'{written:,}', err=errors)

    if buffer:
        pd.DataFrame(buffer).to_csv(class_csv, mode='a',
                                    header=not header_written, index=False)

    elapsed = (time.time()-t0)/60
    return {'class':cls,'n_files':len(files),'n_segs':written,
            'errors':errors,'elapsed_min':round(elapsed,1)}

# --------------------------------------------------------------
# Run only missing classes
# --------------------------------------------------------------
if missing_classes:
    print('='*60)
    print('  RESUMING EXTRACTION FOR MISSING CLASSES')
    print('='*60)
    print(f'  Workers   : {N_WORKERS}')
    print(f'  Features  : 237 per segment')
    print(f'  Batch size: {BATCH_FLUSH}')
    print()

    all_summaries = []
    grand_t0 = time.time()

    for cls in missing_classes:
        print(f'▶  {cls}  ({len(class_files[cls])} files)')
        summ = extract_class(cls, COARSE_MAP[cls],
                             class_files[cls], PART_DIR,
                             N_WORKERS, BATCH_FLUSH)
        all_summaries.append(summ)
        print(f'   ✓  {summ["n_segs"]:>7,} segments  |  '
              f'{summ["errors"]} errors  |  {summ["elapsed_min"]:.1f} min')

    grand_min = (time.time()-grand_t0)/60
    print()
    print(f'Resume extraction total wall-clock: {grand_min:.1f} min')
else:
    print('All classes already have CSVs – no extraction needed.')

  RESUMING EXTRACTION FOR MISSING CLASSES
  Workers   : 4
  Features  : 237 per segment
  Batch size: 50

▶  Nazrul_Geeti_  (254 files)


  Nazrul_Geeti_                 :   0%|          | 0/254 [00:00<?, ?file/s]

   ✓   12,249 segments  |  0 errors  |  114.9 min

Resume extraction total wall-clock: 114.9 min


In [5]:
# ============================================================
# CELL 5 ▸ Merge Per‑Class CSVs (all 8 classes)
# ============================================================
print('Merging all per-class CSVs...')
dfs = []
for cls in CLASS_NAMES:
    cp = os.path.join(PART_DIR, f'feat_{cls}.csv')
    if not os.path.exists(cp):
        print(f'  ⚠️  MISSING: feat_{cls}.csv – cannot merge!')
        continue
    df_c = pd.read_csv(cp)
    dfs.append(df_c)
    print(f'  {cls:<35} {len(df_c):>8,} rows')

if not dfs:
    raise RuntimeError('No CSVs found. Nothing to merge.')

master = pd.concat(dfs, ignore_index=True).fillna(0.0)
master_csv = os.path.join(OUT_DIR, 'features_all.csv')
master.to_csv(master_csv, index=False)

feat_cols = [c for c in master.columns
             if c not in ('filepath','label','coarse','song_id',
                          'seg_idx','seg_duration_s')]
print()
print(f'✅ features_all.csv saved:')
print(f'  Rows (segments) : {len(master):,}')
print(f'  Feature columns : {len(feat_cols)}')
print(f'  Unique files    : {master["filepath"].nunique():,}')
print(f'  File size       : {os.path.getsize(master_csv)/1e6:.1f} MB')

Merging all per-class CSVs...
  Bangla_Band_Collection                10,294 rows
  Bangla_ancholik                        4,480 rows
  Bangla_baby_rhymes                     3,774 rows
  Bangla_rabindra_sangeet                3,813 rows
  Bangla_romantic                        7,213 rows
  Classic_Songs_                         9,299 rows
  Islamic_Melody                         9,273 rows
  Nazrul_Geeti_                         12,249 rows

✅ features_all.csv saved:
  Rows (segments) : 60,395
  Feature columns : 237
  Unique files    : 1,870
  File size       : 277.9 MB


In [6]:
# ============================================================
# CELL 6 ▸ Validation
# ============================================================
print('Validating features_all.csv...\n')
df = pd.read_csv(master_csv)
feat_cols = [c for c in df.columns
             if c not in ('filepath','label','coarse','song_id',
                          'seg_idx','seg_duration_s')]

nan_n = df[feat_cols].isna().sum().sum()
inf_n = np.isinf(df[feat_cols].values).sum()

print(f'CHECK 1 — Feature count     : {len(feat_cols)}  '
      f'({"PASS" if len(feat_cols)==237 else "FAIL expected 237"})')
print(f'CHECK 2 — NaN in features   : {nan_n}  '
      f'({"PASS" if nan_n==0 else "WARN"})')
print(f'CHECK 3 — Inf in features   : {inf_n}  '
      f'({"PASS" if inf_n==0 else "WARN"})')
print()

print('Per-class segment counts:')
print(f'  {"Class":<35} {"Segs":>8}  {"Files":>6}')
print(f'  {"-"*35} {"----":>8}  {"-----":>6}')
for cls in CLASS_NAMES:
    sub = df[df['label']==cls]
    n_s = len(sub); n_f = sub['filepath'].nunique()
    icon = '✅' if n_s > 0 else '❌'
    print(f'  {icon} {cls:<34} {n_s:>8,}  {n_f:>6,}')
print(f'  {"─"*35} {"────":>8}')
print(f'  {"TOTAL":<35} {len(df):>8,}')
print()
print('Imbalance (max/min segments):',
      round(df.groupby("label").size().max() /
            df.groupby("label").size().min(), 2), 'x')

Validating features_all.csv...

CHECK 1 — Feature count     : 237  (PASS)
CHECK 2 — NaN in features   : 0  (PASS)
CHECK 3 — Inf in features   : 0  (PASS)

Per-class segment counts:
  Class                                   Segs   Files
  -----------------------------------     ----   -----
  ✅ Bangla_Band_Collection               10,294     287
  ✅ Bangla_ancholik                       4,480     169
  ✅ Bangla_baby_rhymes                    3,774     153
  ✅ Bangla_rabindra_sangeet               3,813     152
  ✅ Bangla_romantic                       7,213     291
  ✅ Classic_Songs_                        9,299     245
  ✅ Islamic_Melody                        9,273     319
  ✅ Nazrul_Geeti_                        12,249     254
  ───────────────────────────────────     ────
  TOTAL                                 60,395

Imbalance (max/min segments): 3.25 x


In [7]:
# ============================================================
# CELL 7 ▸ ZIP the final output
# ============================================================
zip_path = os.path.join(WORK_DIR, 'part2_features_complete.zip')
print(f'Creating {zip_path} ...')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(master_csv, 'features_all.csv')
    sz = os.path.getsize(master_csv)/1e6
    print(f'  + features_all.csv         {sz:.1f} MB  ← PRIMARY OUTPUT')

    # Optionally include the per-class CSVs
    for cls in CLASS_NAMES:
        cp = os.path.join(PART_DIR, f'feat_{cls}.csv')
        if os.path.exists(cp):
            sz_c = os.path.getsize(cp)/1e6
            zf.write(cp, f'per_class/feat_{cls}.csv')
            print(f'  + per_class/feat_{cls}.csv  {sz_c:.1f} MB')

zip_size = os.path.getsize(zip_path)/1e6
print()
print('='*55)
print('  PART 2 COMPLETE – ALL CLASSES PROCESSED')
print('='*55)
print(f'  ZIP  : {zip_path}  ({zip_size:.1f} MB)')
print(f'  Rows : {len(pd.read_csv(master_csv)):,} total segments')
print('\n  features_all.csv is ready for Part 3.')
print('  Remember to join it with fold_assignments.csv from Part 1 on "filepath".')

Creating /kaggle/working/part2_features_complete.zip ...
  + features_all.csv         277.9 MB  ← PRIMARY OUTPUT
  + per_class/feat_Bangla_Band_Collection.csv  49.5 MB
  + per_class/feat_Bangla_ancholik.csv  21.5 MB
  + per_class/feat_Bangla_baby_rhymes.csv  18.1 MB
  + per_class/feat_Bangla_rabindra_sangeet.csv  18.3 MB
  + per_class/feat_Bangla_romantic.csv  34.6 MB
  + per_class/feat_Classic_Songs_.csv  44.7 MB
  + per_class/feat_Islamic_Melody.csv  44.6 MB
  + per_class/feat_Nazrul_Geeti_.csv  58.7 MB

  PART 2 COMPLETE – ALL CLASSES PROCESSED
  ZIP  : /kaggle/working/part2_features_complete.zip  (254.5 MB)
  Rows : 60,395 total segments

  features_all.csv is ready for Part 3.
  Remember to join it with fold_assignments.csv from Part 1 on "filepath".
